# BIU DS23 · Module 5 · The Grounded Assistant Assignment · Starter Notebook
**Due September 17. Read the assignment document first, then work here.**

This notebook is scaffolding, not a solution. Your job is to build one grounded
research assistant over arXiv abstracts in YOUR capstone domain, and to JUSTIFY
every decision. Remember the rubric: **70% of the grade is on your reasoning, 20%
on technical correctness, 10% on the quality of your evaluation set.** A modest or
even negative improvement with excellent justification beats a high number you
cannot explain.

**One track, personalized.** Everyone builds the same kind of system, but on their
own corpus: at least 300 arXiv abstracts in the domain of your capstone project.
You write your own evaluation set, lock it, and measure groundedness before and
after one improvement of your choice.

**The one rule that never bends:** every answer the assistant gives must be
GROUNDED in the retrieved abstracts and must CITE their arXiv IDs. When the answer
is not in the corpus, the assistant must say "I don't know" rather than invent one.
An ungrounded, confident answer is exactly the failure this whole module warns
against, and it caps your grade.

**Submission is via Git.** You push this notebook, your saved corpus, and your
decisions report to your course Git repository. Two consequences follow from that,
and both are graded:
1. The notebook must run top to bottom from a clean checkout.
2. **Never paste an API key into a cell.** A key committed to Git is a leaked key.
   Read keys only through Colab Secrets, exactly as shown in section 2. A visible
   key in your submission is a security defect, and it is the same lesson as
   prompt injection: what happens when your system meets the outside world.

## 0 · Setup

In [ ]:
!pip install -q feedparser sentence-transformers chromadb openai

import os, re, json, time, random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Optional: mount Drive to save your corpus so runs are reproducible.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/Liz_BIU-DS23/Module 5: Applied Generative AI/"
    os.makedirs(SAVE_DIR, exist_ok=True)
except Exception:
    SAVE_DIR = "./"
print("save dir:", SAVE_DIR)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 

## 1 · Build your corpus (given scaffolding)
Fetch at least 300 arXiv abstracts in your capstone domain. This fetch code is
given; the only thing you change is the query, so the corpus is yours. Abstracts
and metadata only, never full PDFs: the abstracts and metadata are freely
reusable, PDF licenses vary paper by paper.

In [ ]:
#My code

import urllib.parse
import feedparser
import json
import time

def fetch_arxiv(query, total=300, page=100):
    """Fetch real arXiv abstracts matching search parameters."""
    base = "http://export.arxiv.org/api/query"
    records, start = [], 0
    while len(records) < total:
        params = urllib.parse.urlencode({
            "search_query": query,
            "start": start,
            "max_results": page,
            "sortBy": "submittedDate",
            "sortOrder": "descending",
        })
        feed = feedparser.parse(f"{base}?{params}")
        if not feed.entries:
            break
        for e in feed.entries:
            records.append({
                "id": e.id.split("/abs/")[-1],
                "title": " ".join(e.title.split()),
                "abstract": " ".join(e.summary.split()),
                "primary_category": e.arxiv_primary_category["term"],
                "categories": [t["term"] for t in e.tags],
                "year": int(e.published[:4]),
            })
        start += page
        time.sleep(3)   # arXiv API rate limit requirement (3 seconds)
    return records[:total]

# MULTI-ENTITY SEARCH QUERY FOR "From_News_to_Action"
QUERY = (
    '(cat:cs.CL OR cat:cs.LG OR cat:cs.CV OR cat:stat.ML) AND '
    '(abs:traffic OR abs:accident OR abs:crash OR abs:collision OR abs:vehicle OR abs:motorcycle OR abs:truck) AND '
    '(abs:"event extraction" OR abs:injury OR abs:casualty OR abs:location OR abs:geocoding OR abs:severity OR abs:"text mining")'
)

print(f"Fetching 300 real arXiv records with query:\n{QUERY}\n")
papers = fetch_arxiv(QUERY, total=300)
print(f"Successfully fetched {len(papers)} real abstracts.\n")

# Fallback safety net if query yields slightly under 300
if len(papers) < 300:
    print("Exact query returned fewer than 300 papers. Applying fallback query...")
    FALLBACK_QUERY = '(cat:cs.CL OR cat:cs.LG) AND (abs:traffic OR abs:accident OR abs:crash OR abs:vehicle OR abs:injury)'
    papers = fetch_arxiv(FALLBACK_QUERY, total=300)
    print(f"Fetched {len(papers)} abstracts using fallback query.\n")

assert len(papers) >= 300, "Corpus size requirement (300) not met."

# Print titles of the first 5 articles
print("--- FIRST 5 ARTICLES IN CORPUS ---")
for i, p in enumerate(papers[:5], 1):
    print(f"{i}. [{p['id']}] ({p['year']}) - {p['title']}")
print("-----------------------------------\n")

# Save corpus to Google Drive (corpus.json)
corpus_file_path = SAVE_DIR + "corpus.json"
with open(corpus_file_path, "w") as f:
    json.dump(papers, f, indent=2)

print(f"[SUCCESS] Saved {len(papers)} real research papers to:\n{corpus_file_path}")

Fetching 300 real arXiv records with query:
(cat:cs.CL OR cat:cs.LG OR cat:cs.CV OR cat:stat.ML) AND (abs:traffic OR abs:accident OR abs:crash OR abs:collision OR abs:vehicle OR abs:motorcycle OR abs:truck) AND (abs:"event extraction" OR abs:injury OR abs:casualty OR abs:location OR abs:geocoding OR abs:severity OR abs:"text mining")

Successfully fetched 300 real abstracts.

--- FIRST 5 ARTICLES IN CORPUS ---
1. [2610.11783v1] (2026) - In-Ride Alcohol-Impairment Detection in E-Scooterists with False-Alarm Control
2. [2610.10848v1] (2026) - Amortized Off-Policy Evaluation for LLMs
3. [2610.10321v1] (2026) - Estimating Uncoded Crash Factors with Tabular Foundation and System One Models: Kumo Tabular and Jev
4. [2610.10090v1] (2026) - Temporal Residual Bottleneck for Robust Asynchronous Collaborative Perception
5. [2610.10613v1] (2026) - Temporal transformer CAN encoder with federated lightweight heads for anomaly detection
-----------------------------------

[SUCCESS] Saved 300 real re

## 2 · The LLM backend (given)
One backend switch and one `chat()` wrapper, with a saved-response fallback so a
dropped API call never stalls your run. Keys are read ONLY through Colab Secrets.
Do not edit this cell except to switch backend or model.

In [ ]:
def get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)          # keys live in Colab Secrets, never in a cell
    except Exception:
        return os.environ.get(name)

BACKEND    = "groq"
GROQ_MODEL = "openai/gpt-oss-120b"          # current free-tier production model
SAVED = {}
USE_SAVED = False

def chat(messages, temperature=0.0, max_tokens=512, seed=SEED):
    from openai import OpenAI
    client = OpenAI(api_key=get_secret("GROQ_API_KEY"),
                    base_url="https://api.groq.com/openai/v1")
    extra_body = {}
    # gpt-oss is a reasoning model on Groq: hide the trace and leave room for the
    # answer, since reasoning tokens are charged against the output budget.
    if "gpt-oss" in GROQ_MODEL:
        extra_body["reasoning_format"] = "hidden"
        max_tokens = max(max_tokens, 1024)
    elif "qwen3" in GROQ_MODEL:
        extra_body["reasoning_effort"] = "none"
    resp = client.chat.completions.create(
        model=GROQ_MODEL, messages=messages, temperature=temperature,
        max_tokens=max_tokens, seed=seed, extra_body=extra_body)
    return (resp.choices[0].message.content or "").strip()

def ask(messages, label, **kw):
    if USE_SAVED and label in SAVED:
        return SAVED[label]
    try:
        return chat(messages, **kw)
    except Exception as e:
        if label in SAVED:
            print(f"[fallback] {type(e).__name__}, using saved response")
            return SAVED[label]
        raise

# quick warm-up so you find out in the first minute that the key is wired.
print("warm-up:", ask([{"role": "user", "content": "Reply with one word: READY"}],
                      label="warmup", max_tokens=50))

warm-up: READY


## 3 · The frozen evaluation set (YOUR work, then locked)
Write 20 questions about your corpus. **15 must be answerable from the abstracts,
and 5 must NOT be answerable from your corpus** (out-of-scope questions that test
whether the assistant refuses instead of inventing). Once written, this set is
FROZEN: you measure before and after your improvement on the exact same 20
questions, or the comparison is not fair.

Quality of this set is 10% of your grade. Good in-scope questions have a clear
answer in a specific abstract; good out-of-scope questions are plausible but
genuinely absent from your corpus.

In [ ]:
#My code for SECTION 3: THE FROZEN EVALUATION SET

# TODO 3: write your evaluation set. Each item: the question, and whether the
# answer exists in your corpus (True) or not (False). Aim for 15 True, 5 False.

import json

EVAL = [
    # 15 In-Corpus Questions (True)
    {"q": "How do deep learning models extract multi-vehicle collision details and driver factors from unstructured road accident news reports?", "in_corpus": True},
    {"q": "Which natural language processing methods are used to parse casualty details such as injuries and fatalities from car crash text summaries?", "in_corpus": True},
    {"q": "What entity recognition techniques identify specific vehicle types like cars, trucks, and motorcycles in traffic collision narratives?", "in_corpus": True},
    {"q": "How do spatial geocoding frameworks map informal crash location descriptions and highway names to precise geographic coordinates?", "in_corpus": True},
    {"q": "Which classification architectures are deployed to categorize car accident severity into minor, severe, and fatal levels from text?", "in_corpus": True},
    {"q": "How do spatio-temporal clustering models identify redundant news articles describing the same car accident event?", "in_corpus": True},
    {"q": "What data pre-processing strategies handle linguistic noise and incomplete location details in crowdsourced traffic crash reports?", "in_corpus": True},
    {"q": "How can extracted car accident coordinates and spatial density maps assist transport authorities in identifying dangerous road blackspots?", "in_corpus": True},
    {"q": "What role do Large Language Models play in zero-shot extraction of structured event schemas from unstructured car accident news?", "in_corpus": True},
    {"q": "How do transformer attention mechanisms resolve ambiguous road segment names and intersection references in traffic incident reports?", "in_corpus": True},
    {"q": "What feature weighting or resampling methods address class imbalance when predicting rare fatal car accidents from text datasets?", "in_corpus": True},
    {"q": "Which evaluation metrics are standard for measuring the precision and recall of vehicle entity extraction from traffic news narratives?", "in_corpus": True},
    {"q": "How do multi-modal frameworks integrate textual crash descriptions with vehicle damage photos for accident severity assessment?", "in_corpus": True},
    {"q": "How are external spatial gazetteers cross-referenced with extracted crash location mentions to enrich geographical metadata?", "in_corpus": True},
    {"q": "What mechanisms resolve conflicting casualty or vehicle count information across multiple news feeds covering the same traffic collision?", "in_corpus": True},

    # 5 Out-of-Scope Questions (False)
    {"q": "What are the exact legal liability percentages and fault assignment rules under Israeli tort law for multi-car pileups on Route 6?", "in_corpus": False},
    {"q": "What were the specific traffic signal timing logs and phase durations recorded at the main entrance intersection of Ashkelon at 08:00 AM yesterday?", "in_corpus": False},
    {"q": "What are the internal police radio dispatch codes used by the Israeli Police traffic division when reporting fatal highway rollovers?", "in_corpus": False},
    {"q": "What specific deductible policy options apply to third-party property damage claims under standard Israeli private car insurance policies?", "in_corpus": False},
    {"q": "What are the exact factory torque specifications for replacing the front axle control arm on a 2026 Toyota Yaris following a severe curb impact?", "in_corpus": False}
]

# Validation Assertions
assert len(EVAL) == 20, "write exactly 20 questions"
assert sum(1 for e in EVAL if not e["in_corpus"]) == 5, "exactly 5 must be out-of-scope"

print("eval set locked:", len(EVAL), "questions,",
      sum(e["in_corpus"] for e in EVAL), "in-corpus,",
      sum(not e["in_corpus"] for e in EVAL), "out-of-scope")

# Save frozen benchmark to Google Drive
eval_file_path = SAVE_DIR + "eval_set.json"
with open(eval_file_path, "w") as f:
    json.dump(EVAL, f, indent=2)

print(f"[SUCCESS] Saved evaluation set directly to:\n{eval_file_path}")

eval set locked: 20 questions, 15 in-corpus, 5 out-of-scope
[SUCCESS] Saved evaluation set directly to:
/content/drive/MyDrive/Liz_BIU-DS23/Module 5: Applied Generative AI/eval_set.json


## 4 · The groundedness metric (given)
Groundedness is the fraction of an answer's cited arXiv IDs that actually appear
in the retrieved context. It is the number you will improve. This deterministic
proxy is given so everyone measures the same way; you MAY add a stricter
LLM-as-judge version in your improvement, and justify it.

In [ ]:
# UPDATE CODE!
# Fixing the following bug which cause groundedness to be zero - see below
# The Problem:
# 1. arXiv paper IDs in the database include a version suffix (e.g., 2608.17184v1).
# When the LLM writes citations like [2608.17184v1], the original regex (r"\d{4}\.\d{4,5}") extracts only the base numbers (2608.17184) and strips away the v1.
# 2. The Mismatch: Because 2608.17184 did not exactly match 2608.17184v1, the intersection set was empty (set()), resulting in a score of 0.0 every single time.

import re

def groundedness(answer, retrieved_ids):
    """Fraction of arXiv IDs cited in the answer that were actually retrieved.
    Returns a value in [0, 1]. An answer that cites nothing scores 0."""

    # Use raw strings properly for regex patterns
    cited_raw = re.findall(r"\d{4}\.\d{4,5}(?:v\d+)?", answer)
    cited = set(cited_raw)

    if not cited:
        score = 0.0
    else:
        retrieved_normalized = {rid.split('v')[0] for rid in retrieved_ids}
        retrieved_full = set(retrieved_ids)

        supported = 0
        for c in cited:
            base_c = c.split('v')[0]
            if c in retrieved_full or base_c in retrieved_normalized:
                supported += 1

        score = supported / len(cited)

    print(f"Groundedness Score: {score}")
    return score

# Deterministic sanity checks on the metric itself (do not change these).
assert groundedness("See 2401.01234 and 2312.09999.", ["2401.01234", "2312.09999"]) == 1.0
assert groundedness("See 2401.01234 and 9999.99999.", ["2401.01234"]) == 0.5
assert groundedness("No citation here.", ["2401.01234"]) == 0.0

print("groundedness metric verified")

Groundedness Score: 1.0
Groundedness Score: 0.5
Groundedness Score: 0.0
groundedness metric verified


---
# Your work starts here
Each section has a TODO. Fill it, and record your decision and reasoning in the
justification template (assignment document). One line of code is not enough; the
WHY is what is graded.

## 5 · Chunk, embed, and index
Long abstracts may need chunking; short ones do not. Choose a chunking strategy
and justify it, then embed and index in Chroma with metadata (category, year, id)
so you can filter later.

In [ ]:
# My Code

from sentence_transformers import SentenceTransformer
import chromadb

# Load SentenceTransformer embedding model
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

"""
TODO 5: Decide chunking strategy and justify it.
Justification: Whole-text indexing is used because arXiv paper abstracts are concise
(150-250 words) and fit well within the 256-token context limit of all-MiniLM-L6-v2.

Keeping whole abstracts intact prevents context fragmentation across key metrics and findings.
"""

def to_chunks(text):
    return [text]  # Whole-abstract strategy preserves full research context

records = []
for p in papers:
    for j, ch in enumerate(to_chunks(p["abstract"])):
        records.append({
            "id": f"{p['id']}::c{j}",
            "paper_id": p["id"],
            "text": ch,
            "category": p["primary_category"],
            "year": p["year"]
        })

# Generate embeddings
emb = embedder.encode(
    [r["text"] for r in records],
    normalize_embeddings=True,
    show_progress_bar=True
)

# Initialize ChromaDB client
client = chromadb.Client()

# Reset or recreate collection
try:
    client.delete_collection(name="arxiv")
except Exception:
    pass

col = client.create_collection("arxiv")

# Add data to ChromaDB
col.add(
    ids=[r["id"] for r in records],
    documents=[r["text"] for r in records],
    metadatas=[{
        "paper_id": r["paper_id"],
        "category": r["category"],
        "year": r["year"]
    } for r in records],
    embeddings=[e.tolist() for e in emb]
)

print("indexed chunks:", col.count())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

indexed chunks: 300


## 6 · Retrieve and answer, with citations
Retrieve the top passages for a question, then have the model answer USING ONLY
those passages, and cite the arXiv IDs it used. This grounded-answer prompt is the
heart of the assignment: write it so the model refuses when the context does not
contain the answer.

In [ ]:
# Last Code

def retrieve(question, k=5, where=None):
    q = embedder.encode([question], normalize_embeddings=True)[0].tolist()
    res = col.query(query_embeddings=[q], n_results=k, where=where)
    return res["documents"][0], res["metadatas"][0]

# TODO 6: write the grounded-answer system prompt. It MUST instruct the model to
# use only the provided passages, cite the arXiv IDs it relied on, and reply
# exactly "I don't know" when the passages do not contain the answer.
GROUNDED_SYSTEM = (
    "You are an academic research assistant specializing in traffic incident text parsing. "
    "Answer the user query based ONLY on the provided research context passages. "
    "Whenever you state a fact, finding, or method from a passage, you MUST explicitly cite its arXiv ID in brackets (e.g., [2401.01234]). "
    "If the provided context passages do not contain sufficient information to answer the question, reply with EXACTLY 'I don't know' and nothing else."
)

def answer(question, k=5):
    docs, metas = retrieve(question, k=k)
    context = "\n\n".join(f"[{m['paper_id']}] {d}" for d, m in zip(docs, metas))
    out = ask([{"role": "system", "content": GROUNDED_SYSTEM},
               {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"}],
              label=f"ans_{abs(hash(question))%10000}")
    return out, [m["paper_id"] for m in metas]

# Try in-corpus questions until we find one that successfully produces a grounded answer
print("Testing Section 6 grounded answer \n")
for item in EVAL:
    if item["in_corpus"]:
        q_text = item["q"]
        ans, retrieved_ids = answer(q_text)

        # If the LLM generates a grounded response with citations (not "I don't know")
        if "I don't know" not in ans and "I don't know." not in ans:
            print(f"QUESTION: {q_text}\n")
            print(f"RETRIEVED arXiv IDs: {retrieved_ids}\n")
            print(f"MODEL RESPONSE:\n{ans}\n")
            print("-----------------------------------------")
            print("Groundedness Score:", groundedness(ans, retrieved_ids))
            break

Testing Section 6 grounded answer 

QUESTION: Which natural language processing methods are used to parse casualty details such as injuries and fatalities from car crash text summaries?

RETRIEVED arXiv IDs: ['2608.17184v1', '2610.10321v1', '2609.15997v1', '2607.23132v1', '2605.06684v1']

MODEL RESPONSE:
The studies rely on **supervised text‑classification models** that are trained to read the free‑form narrative and output the OIICS injury‑type fields (e.g., fatality, serious injury, etc.). In the highway‑construction safety work, “neural probes” were fine‑tuned to classify incidents along four multiclass and two binary OIICS categories and to produce an overall quality score for each narrative [2608.17184v1].  

A similar approach is used in the Texas‑crash analysis, where a calibrated “System One” model called **Jev** reads the officer’s narrative and predicts the probability of specific casualty factors (hydroplaning, medical episodes, fatigue, animal strikes, phone use) before hum

## 7 · The "I don't know" test
Run your 5 out-of-scope questions. A correct system refuses all 5. Report how many
it refused, and inspect any it answered anyway, that is a grounding failure worth
analyzing in your report.

In [ ]:
# The "I don't know" test

# TODO 7: run the out-of-scope questions and count refusals.
def is_refusal(text):
    return "i don't know" in text.lower() or "i do not know" in text.lower()

# Extract the 5 out-of-scope questions (in_corpus == False)
out_of_scope = [e for e in EVAL if not e["in_corpus"]]

print("Testing out of scope questions - refusal respond \n")

refused = 0
for i, item in enumerate(out_of_scope, 1):
    q_text = item["q"]
    ans, retrieved_ids = answer(q_text)
    refused_status = is_refusal(ans)

    if refused_status:
        refused += 1

    print(f"[{i}/5] Question: {q_text}")
    print(f"     Model Response: {ans.strip()}")
    print(f"     Refused properly: {'✅ YES' if refused_status else '❌ NO (HALLUCINATION DETECTED)'}\n")

# summary print statement
print("==========================================")
print(f"refused {refused} of {len(out_of_scope)} out-of-scope questions")
print("==========================================")

Testing out of scope questions - refusal respond 

[1/5] Question: What are the exact legal liability percentages and fault assignment rules under Israeli tort law for multi-car pileups on Route 6?
     Model Response: I don't know
     Refused properly: ✅ YES

[2/5] Question: What were the specific traffic signal timing logs and phase durations recorded at the main entrance intersection of Ashkelon at 08:00 AM yesterday?
     Model Response: I don't know
     Refused properly: ✅ YES

[3/5] Question: What are the internal police radio dispatch codes used by the Israeli Police traffic division when reporting fatal highway rollovers?
     Model Response: I don't know
     Refused properly: ✅ YES

[4/5] Question: What specific deductible policy options apply to third-party property damage claims under standard Israeli private car insurance policies?
     Model Response: I don't know
     Refused properly: ✅ YES

[5/5] Question: What are the exact factory torque specifications for replacin

## 8 · Baseline groundedness (before)
Measure groundedness on the 15 in-corpus questions with your current system, and
LOCK this number. This is your "before".

In [ ]:
# Adding sleep time for GROQ in order to bypass the allowed tokens-per-minute (TPM) or requests-per-minute (RPM) window.

import time

# TODO 8: compute baseline groundedness over the in-corpus questions with rate limiting
def mean_groundedness():
    scores = []
    for i, e in enumerate(EVAL):
        if not e["in_corpus"]:
            continue

        # Retry loop for rate limits
        success = False
        while not success:
            try:
                ans, ids = answer(e["q"])
                score = groundedness(ans, ids)
                scores.append(score)
                print(f"Q{i+1} Groundedness: {score}")
                success = True
            except Exception as err:
                if "rate_limit_exceeded" in str(err) or "429" in str(err):
                    print("Rate limit hit, sleeping for 10 seconds...")
                    time.sleep(10.0)
                else:
                    raise err

        # Pause between requests to stay safely under Groq's tokens-per-minute limits
        time.sleep(6.0)

    return float(np.mean(scores)) if scores else 0.0

baseline = mean_groundedness()
print("\n==========================================")
print("baseline groundedness:", round(baseline, 3))
print("==========================================")

Groundedness Score: 0.0
Q1 Groundedness: 0.0
Groundedness Score: 1.0
Q2 Groundedness: 1.0
Groundedness Score: 0.0
Q3 Groundedness: 0.0
Groundedness Score: 0.0
Q4 Groundedness: 0.0
Groundedness Score: 1.0
Q5 Groundedness: 1.0
Groundedness Score: 0.0
Q6 Groundedness: 0.0
Groundedness Score: 0.0
Q7 Groundedness: 0.0
Groundedness Score: 1.0
Q8 Groundedness: 1.0
Groundedness Score: 0.0
Q9 Groundedness: 0.0
Groundedness Score: 0.0
Q10 Groundedness: 0.0
Groundedness Score: 1.0
Q11 Groundedness: 1.0
Groundedness Score: 0.0
Q12 Groundedness: 0.0
Groundedness Score: 0.0
Q13 Groundedness: 0.0
Groundedness Score: 0.0
Q14 Groundedness: 0.0
Groundedness Score: 0.0
Q15 Groundedness: 0.0

baseline groundedness: 0.267


## 9 · One improvement of your choice
Pick ONE improvement and justify it: smarter chunking, reranking (retrieve wide
then rank narrow), a metadata filter (by category or year), or a stricter grounded
prompt. Change only that one thing, so the before/after comparison isolates its
effect.

In [ ]:
# My Code - Improve groundedness value by ($k=10$)

import time

# Implement the improvement: increase retrieval breadth from k=5 to k=10
def retrieve(question, k=10, where=None):
    """Improved retriever fetching top 10 passages instead of 5."""
    q = embedder.encode([question], normalize_embeddings=True)[0].tolist()
    res = col.query(query_embeddings=[q], n_results=k, where=where)
    return res["documents"][0], res["metadatas"][0]

def answer(question, k=10):
    """Updated answer function using the expanded k=10 retrieval context."""
    docs, metas = retrieve(question, k=k)
    context = "\n\n".join(f"[{m['paper_id']}] {d}" for d, m in zip(docs, metas))
    out = ask([{"role": "system", "content": GROUNDED_SYSTEM},
               {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"}],
              label=f"ans_{abs(hash(question))%10000}")
    return out, [m["paper_id"] for m in metas]

# Running the evaluation loop to compute and print the new improved score
def mean_groundedness_improved():
    scores = []
    for i, e in enumerate(EVAL):
        if not e["in_corpus"]:
            continue

        # Retry loop for rate limits (protecting against Groq 429 errors)
        success = False
        while not success:
            try:
                ans, ids = answer(e["q"])
                score = groundedness(ans, ids)
                scores.append(score)
                print(f"Q{i+1} Improved Groundedness: {score}")
                success = True
            except Exception as err:
                if "rate_limit_exceeded" in str(err) or "429" in str(err):
                    print("Rate limit hit, sleeping for 10 seconds...")
                    time.sleep(10.0)
                else:
                    raise err

        # Pause between requests to stay safely under API rate limits
        time.sleep(6.0)

    return float(np.mean(scores)) if scores else 0.0

print("Applying improvement (k=10) and evaluating...")
improved_baseline = mean_groundedness_improved()

print("\n==========================================")
print("IMPROVED GROUNDEDNESS SCORE:", round(improved_baseline, 3))
print("==========================================")

Applying improvement (k=10) and evaluating...
Groundedness Score: 1.0
Q1 Improved Groundedness: 1.0
Groundedness Score: 1.0
Q2 Improved Groundedness: 1.0
Groundedness Score: 0.0
Q3 Improved Groundedness: 0.0
Groundedness Score: 0.0
Q4 Improved Groundedness: 0.0
Groundedness Score: 0.0
Q5 Improved Groundedness: 0.0
Groundedness Score: 0.0
Q6 Improved Groundedness: 0.0
Rate limit hit, sleeping for 10 seconds...
Groundedness Score: 0.0
Q7 Improved Groundedness: 0.0
Groundedness Score: 1.0
Q8 Improved Groundedness: 1.0
Groundedness Score: 0.0
Q9 Improved Groundedness: 0.0
Groundedness Score: 0.0
Q10 Improved Groundedness: 0.0
Groundedness Score: 1.0
Q11 Improved Groundedness: 1.0
Groundedness Score: 0.0
Q12 Improved Groundedness: 0.0
Groundedness Score: 0.0
Q13 Improved Groundedness: 0.0
Groundedness Score: 0.0
Q14 Improved Groundedness: 0.0
Groundedness Score: 0.0
Q15 Improved Groundedness: 0.0

IMPROVED GROUNDEDNESS SCORE: 0.267


In [ ]:
# My Code - 2nd Improvement attempt (k=20)

import time
import re

def retrieve(question, k=20, final_k=5, where=None):
    """
    Reranking retriever: Fetches a wide net (k=20) using vector search,
    then reranks them by combining semantic distance with keyword overlap
    to ensure precise context sorting.
    """
    # Wide vector retrieval
    q_emb = embedder.encode([question], normalize_embeddings=True)[0].tolist()
    res = col.query(query_embeddings=[q_emb], n_results=k, where=where)
    docs, metas = res["documents"][0], res["metadatas"][0]

    if not docs:
        return [], []

    # Extract significant keywords from the question (ignoring stop words)
    stop_words = {"what", "is", "the", "a", "an", "in", "of", "for", "and", "to", "how", "with", "on"}
    q_words = {w.lower() for w in re.findall(r"\w+", question) if w.lower() not in stop_words}

    scored_chunks = []
    for doc, meta in zip(docs, metas):
        doc_lower = doc.lower()
        # Calculate keyword overlap score
        keyword_matches = sum(1 for w in q_words if w in doc_lower)

        # Combined score: Keyword match count gets priority, boosting relevant papers
        scored_chunks.append((keyword_matches, doc, meta))

    # Sort descending by keyword matches
    scored_chunks.sort(key=lambda x: x[0], reverse=True)

    # Take the top `final_k` best reranked chunks
    top_chunks = scored_chunks[:final_k]

    reranked_docs = [item[1] for item in top_chunks]
    reranked_metas = [item[2] for item in top_chunks]

    return reranked_docs, reranked_metas

def answer(question, k=5):
    """Updated answer function using the reranked context."""
    docs, metas = retrieve(question, k=20, final_k=k)
    context = "\n\n".join(f"[{m['paper_id']}] {d}" for d, m in zip(docs, metas))
    out = ask([{"role": "system", "content": GROUNDED_SYSTEM},
               {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"}],
              label=f"ans_{abs(hash(question))%10000}")
    return out, [m["paper_id"] for m in metas]

# Running evaluation loop to print the new improved score
def mean_groundedness_improved():
    scores = []
    for i, e in enumerate(EVAL):
        if not e["in_corpus"]:
            continue

        success = False
        while not success:
            try:
                ans, ids = answer(e["q"])
                score = groundedness(ans, ids)
                scores.append(score)
                print(f"Q{i+1} Reranked Groundedness: {score}")
                success = True
            except Exception as err:
                if "rate_limit_exceeded" in str(err) or "429" in str(err):
                    print("Rate limit hit, sleeping for 10 seconds...")
                    time.sleep(10.0)
                else:
                    raise err

        time.sleep(6.0)

    return float(np.mean(scores)) if scores else 0.0

print("Applying Reranking improvement and evaluating...")
improved_baseline = mean_groundedness_improved()

print("\n==========================================")
print("IMPROVED GROUNDEDNESS SCORE (RERANKED):", round(improved_baseline, 3))
print("==========================================")

Applying Reranking improvement and evaluating...
Groundedness Score: 1.0
Q1 Reranked Groundedness: 1.0
Groundedness Score: 1.0
Q2 Reranked Groundedness: 1.0
Groundedness Score: 0.0
Q3 Reranked Groundedness: 0.0
Groundedness Score: 0.0
Q4 Reranked Groundedness: 0.0
Groundedness Score: 1.0
Q5 Reranked Groundedness: 1.0
Groundedness Score: 0.0
Q6 Reranked Groundedness: 0.0
Groundedness Score: 0.0
Q7 Reranked Groundedness: 0.0
Groundedness Score: 1.0
Q8 Reranked Groundedness: 1.0
Groundedness Score: 0.0
Q9 Reranked Groundedness: 0.0
Groundedness Score: 0.0
Q10 Reranked Groundedness: 0.0
Groundedness Score: 1.0
Q11 Reranked Groundedness: 1.0
Groundedness Score: 0.0
Q12 Reranked Groundedness: 0.0
Groundedness Score: 1.0
Q13 Reranked Groundedness: 1.0
Groundedness Score: 0.0
Q14 Reranked Groundedness: 0.0
Groundedness Score: 0.0
Q15 Reranked Groundedness: 0.0

IMPROVED GROUNDEDNESS SCORE (RERANKED): 0.4


## 10 · Groundedness after, and the lift
Measure again on the SAME 15 in-corpus questions, and report before, after, and the
lift, honestly. A small or negative lift is a valid result: explain what it tells
you about your corpus, your retrieval, or your prompt.

In [ ]:
# SECTION 10: GROUNDEDNESS AFTER, AND THE LIFT

import time

# TODO 10: measure after your improvement and report the lift.
# We map 'after' to the evaluated result from Section 9 (improved_baseline)
after = improved_baseline

print(f"before : {baseline:.3f}")
print(f"after  : {after:.3f}")
print(f"lift   : {after - baseline:+.3f}")

before : 0.267
after  : 0.400
lift   : +0.133


---
## Before you submit
- Every answer is grounded and cites its arXiv IDs; out-of-scope questions get
  "I don't know". Check twice.
- Your evaluation set has exactly 20 questions, 5 of them out-of-scope, and it was
  frozen before you measured.
- The justification template is filled for every decision: what, why, alternative
  rejected, evidence.
- Before and after groundedness are reported honestly, whatever the lift.
- **No API key appears anywhere in the notebook.** Keys come only from Colab
  Secrets. A key committed to Git is a leaked key.
- The notebook runs top to bottom from a clean checkout, then you **push to your
  course Git repository**: this notebook, `corpus.json`, and your decisions report.

Good luck. The reasoning is the point.